In [ ]:
import lightgbm as lgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import average_precision_score

In [ ]:
# Load the CICIDS 2017 dataset (assuming the dataset is in CSV format)
# Make sure to load your dataset path here
df = pd.read_csv('cicids2017_cleaned.csv')

In [ ]:
# Assuming the target column is 'Label' and features are the rest
X = df.drop(columns=['Label'])
y = df['Label']

# Encode labels if necessary
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [ ]:
# Split the dataset into train, validation, and test sets (70%, 15%, 15%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [ ]:
# Initialize LightGBM model
lgb_model = lgb.LGBMClassifier(num_class=len(np.unique(y_encoded)), objective='multiclass', metric='multi_logloss')

# Train the model
lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, verbose=10)

# Make predictions
y_pred = lgb_model.predict(X_test)
y_pred_prob = lgb_model.predict_proba(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}')




In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()



In [ ]:
# Precision, Recall, F1 Score
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')



In [ ]:
# ROC Curve for multiclass (One-vs-Rest approach)
fpr, tpr, roc_auc = {}, {}, {}
n_classes = len(np.unique(y_encoded))

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test == i, y_pred_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC curve
plt.figure(figsize=(10, 7))
for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], label=f'Class {label_encoder.classes_[i]} (AUC = {roc_auc[i]:.2f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curve for Multiclass')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.show()



In [ ]:
# Precision-Recall Curve
plt.figure(figsize=(10, 7))
for i in range(n_classes):
    precision_vals, recall_vals, _ = precision_recall_curve(y_test == i, y_pred_prob[:, i])
    avg_precision = average_precision_score(y_test == i, y_pred_prob[:, i])
    plt.plot(recall_vals, precision_vals, label=f'Class {label_encoder.classes_[i]} (AP = {avg_precision:.2f})')

plt.title('Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend(loc='best')
plt.show()